# LeetCode 2256 — Minimum Average Difference bằng Python

Link đề: https://leetcode.com/problems/minimum-average-difference/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minimumAverageDifference(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc chính xác định nghĩa: chỉ số `i` chia mảng thành **`i + 1` phần tử đầu** và **`n − i − 1` phần tử cuối**; trung bình là **chia lấy nguyên**; trung bình của 0 phần tử là 0.
2. Duy trì **tổng tiền tố** và **tổng hậu tố** khi quét để tính mỗi chỉ số trong O(1).
3. Xử lý đúng ca **chia cho 0** (chỉ số cuối) và quy tắc **hòa lấy chỉ số nhỏ nhất** (`<` chứ không phải `<=`).
4. Hiểu khác biệt giữa `//` (làm tròn xuống) và `int(a / b)` (làm tròn về 0, và dùng float) — dù ở bài này số không âm.
5. Viết test đối chiếu với vét cạn O(n²), thêm test với số lớn để thấy tổng có thể tới 10¹⁰.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` (chỉ số từ 0). **Hiệu trung bình** của chỉ số `i` là

`| ⌊(nums[0] + … + nums[i]) / (i + 1)⌋ − ⌊(nums[i+1] + … + nums[n−1]) / (n − i − 1)⌋ |`

với quy ước trung bình của 0 phần tử là 0 (khi `i = n − 1`).

Trả về chỉ số có hiệu trung bình **nhỏ nhất**; nếu hòa, trả chỉ số **nhỏ nhất**.

**Ví dụ 1:** `nums = [2,5,3,9,5,3]` → `3`. Các hiệu: `3, 2, 2, 0, 1, 4`.

**Ví dụ 2:** `nums = [0]` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁵`.

In [1]:
from typing import List
from itertools import accumulate
from fractions import Fraction
import random, time

examples = [([2, 5, 3, 9, 5, 3], 3), ([0], 0)]
def avg_diff(nums, i):
    n = len(nums)
    left = sum(nums[:i + 1]) // (i + 1)
    right = sum(nums[i + 1:]) // (n - i - 1) if n - i - 1 else 0
    return abs(left - right)

for nums, exp in examples:
    print(nums, "hiệu theo i:", [avg_diff(nums, i) for i in range(len(nums))], "-> kỳ vọng", exp)

[2, 5, 3, 9, 5, 3] hiệu theo i: [3, 2, 2, 0, 1, 4] -> kỳ vọng 3
[0] hiệu theo i: [0] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Chia lấy nguyên `//` và làm tròn

Với số **không âm**, `a // b` = phần nguyên của `a / b` (làm tròn xuống). Với số âm, `//` làm tròn **xuống** (về −∞) còn `int(a / b)` làm tròn **về 0** — hai cách khác nhau. Đề này chỉ có số không âm nhưng nên biết.

In [2]:
for a, b in [(7, 2), (10, 3), (-7, 2)]:
    print(f"{a} // {b} = {a // b}, int({a} / {b}) = {int(a / b)}")

7 // 2 = 3, int(7 / 2) = 3
10 // 3 = 3, int(10 / 3) = 3
-7 // 2 = -4, int(-7 / 2) = -3


### 3.2 Tổng tiền tố và hậu tố

`pre[i] = nums[0] + … + nums[i − 1]`; tổng `i + 1` phần tử đầu = `pre[i + 1]`; tổng phần còn lại = `total − pre[i + 1]`.

In [3]:
nums = [2, 5, 3, 9, 5, 3]
pre = list(accumulate(nums, initial=0))
total = pre[-1]
print("pre =", pre, "total =", total)
for i in range(len(nums)):
    print(f"i={i}: đầu {pre[i + 1]} / {i + 1}, cuối {total - pre[i + 1]} / {len(nums) - i - 1}")

pre = [0, 2, 7, 10, 19, 24, 27] total = 27
i=0: đầu 2 / 1, cuối 25 / 5
i=1: đầu 7 / 2, cuối 20 / 4
i=2: đầu 10 / 3, cuối 17 / 3
i=3: đầu 19 / 4, cuối 8 / 2
i=4: đầu 24 / 5, cuối 3 / 1
i=5: đầu 27 / 6, cuối 0 / 0


### 3.3 Tìm chỉ số nhỏ nhất đạt min: `min(range(n), key=f)`

`min` trả về phần tử **đầu tiên** đạt giá trị nhỏ nhất ⇒ tự động thỏa "hòa lấy chỉ số nhỏ nhất".

In [4]:
vals = [3, 2, 2, 0, 1, 0]
print(min(range(len(vals)), key=lambda i: vals[i]), "(chỉ số đầu tiên có giá trị 0)")

3 (chỉ số đầu tiên có giá trị 0)


### 3.4 Tránh chia cho 0 bằng biểu thức điều kiện

In [5]:
def safe_avg(s, k):
    return s // k if k else 0
print(safe_avg(10, 3), safe_avg(0, 0))
try:
    10 // 0
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

3 0
ZeroDivisionError: integer division or modulo by zero


### 3.5 `round()` của Python làm tròn "về số chẵn gần nhất"

`round(2.5) = 2`, `round(3.5) = 4` (banker's rounding) — vừa **không** phải làm tròn xuống, vừa không phải làm tròn "nửa lên" như học ở trường. Đề yêu cầu làm tròn **xuống** ⇒ chỉ dùng `//`. Ngoài ra `left / (i + 1)` tạo `float`, với tổng lớn (tới 10¹⁰) vẫn chính xác, nhưng không cần thiết.

In [6]:
for v in [0.5, 1.5, 2.5, 3.5, 3.4999]:
    print(f"round({v}) = {round(v)}, int({v}) = {int(v)}, {v} // 1 = {v // 1}")

round(0.5) = 0, int(0.5) = 0, 0.5 // 1 = 0.0
round(1.5) = 2, int(1.5) = 1, 1.5 // 1 = 1.0
round(2.5) = 2, int(2.5) = 2, 2.5 // 1 = 2.0
round(3.5) = 4, int(3.5) = 3, 3.5 // 1 = 3.0
round(3.4999) = 3, int(3.4999) = 3, 3.4999 // 1 = 3.0


## 4. Ý tưởng nền tảng / trực giác

Mỗi chỉ số `i` đặt một "vách" ngay **sau** phần tử `i`: bên trái `i + 1` phần tử (luôn ≥ 1), bên phải `n − i − 1` phần tử (có thể 0).

Tính tay ví dụ 1 (`total = 27`):

| i | tổng trái | đếm trái | TB trái | tổng phải | đếm phải | TB phải | hiệu |
|---|---|---|---|---|---|---|---|
| 0 | 2 | 1 | 2 | 25 | 5 | 5 | 3 |
| 1 | 7 | 2 | 3 | 20 | 4 | 5 | 2 |
| 2 | 10 | 3 | 3 | 17 | 3 | 5 | 2 |
| 3 | 19 | 4 | 4 | 8 | 2 | 4 | **0** |
| 4 | 24 | 5 | 4 | 3 | 1 | 3 | 1 |
| 5 | 27 | 6 | 4 | 0 | 0 | 0 (quy ước) | 4 |

Nhận xét: khi vách dịch sang phải một bước, tổng trái **cộng** `nums[i]`, tổng phải **trừ** `nums[i]` — chỉ cần hai biến, không cần tính lại tổng.

**Lưu ý về làm tròn:** trung bình làm tròn **trước** khi lấy hiệu. Ví dụ `i = 1`: `7/2 = 3,5 → 3`, `20/4 = 5` ⇒ hiệu 2 (không phải 1,5).

In [7]:
nums = [2, 5, 3, 9, 5, 3]; n = len(nums); left = 0; right = sum(nums)
for i, x in enumerate(nums):
    left += x; right -= x
    a = left // (i + 1); b = right // (n - i - 1) if n - i - 1 else 0
    print(f"i={i}: trái {left}/{i+1} -> {a}; phải {right}/{n-i-1} -> {b}; hiệu {abs(a - b)}")

i=0: trái 2/1 -> 2; phải 25/5 -> 5; hiệu 3
i=1: trái 7/2 -> 3; phải 20/4 -> 5; hiệu 2
i=2: trái 10/3 -> 3; phải 17/3 -> 5; hiệu 2
i=3: trái 19/4 -> 4; phải 8/2 -> 4; hiệu 0
i=4: trái 24/5 -> 4; phải 3/1 -> 3; hiệu 1
i=5: trái 27/6 -> 4; phải 0/0 -> 0; hiệu 4


### 4.1 Hiệu trung bình **không đơn điệu** — minh họa

Nhiều người hy vọng hiệu giảm dần rồi tăng dần (hình chữ V) để tìm nhị phân / tam phân. Không đúng: các giá trị có thể dao động lên xuống tùy dữ liệu. Ta vẽ hiệu theo `i` cho một mảng ngẫu nhiên:

In [8]:
rng0 = random.Random(3)
demo = [rng0.randint(0, 20) for _ in range(16)]
vals = [avg_diff(demo, i) for i in range(len(demo))]
print("nums :", demo)
print("hiệu :", vals)
top = max(vals)
for level in range(top, -1, -max(1, top // 8)):
    print(f"{level:>3} | " + "".join("#" if v >= level else " " for v in vals))

nums : [7, 18, 17, 4, 11, 19, 15, 20, 18, 2, 19, 0, 15, 8, 17, 7]
hiệu : [5, 0, 3, 1, 1, 0, 2, 3, 5, 2, 4, 1, 2, 0, 5, 12]
 12 |                #
 11 |                #
 10 |                #
  9 |                #
  8 |                #
  7 |                #
  6 |                #
  5 | #       #     ##
  4 | #       # #   ##
  3 | # #    ## #   ##
  2 | # #   ##### # ##
  1 | # ### ####### ##
  0 | ################


Đồ thị lởm chởm, có nhiều cực tiểu địa phương ⇒ phải duyệt **mọi** chỉ số. May mắn là mỗi chỉ số chỉ tốn O(1) nhờ tổng chạy.

## 5. Cách 1 — Brute Force / baseline: tính lại hai tổng cho mỗi chỉ số

In [9]:
def brute(nums):
    n = len(nums); best, ans = None, 0
    for i in range(n):
        left = sum(nums[:i + 1]) // (i + 1)
        right = sum(nums[i + 1:]) // (n - i - 1) if n - i - 1 else 0
        d = abs(left - right)
        if best is None or d < best:
            best, ans = d, i
    return ans

for nums, exp in examples:
    got = brute(nums)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp:** mỗi chỉ số tốn O(n) để cộng hai lát cắt ⇒ **O(n²)**; `n = 10⁵` ⇒ 10¹⁰ phép.

**Nút thắt:** tổng của `nums[:i+1]` và `nums[:i+2]` chỉ khác một phần tử. Cộng dồn ⇒ O(1) mỗi bước.

In [10]:
for n in [2000, 6000]:
    a = [random.randint(0, 10**5) for _ in range(n)]
    t = time.perf_counter(); brute(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=2000: 26 ms


n=6000: 223 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `L_i` = tổng `i + 1` phần tử đầu, `R_i = total − L_i`. Ta có `L_i = L_{i−1} + nums[i]`. Do đó quét một lượt với hai biến `left`, `right`:
- `left += nums[i]`, `right −= nums[i]`.
- `a = left // (i + 1)`; `b = right // (n − i − 1)` nếu `n − i − 1 > 0`, ngược lại `b = 0`.
- **Nếu** `|a − b| < best` **thì** cập nhật `best` và `ans = i` (dùng `<` để giữ chỉ số **nhỏ nhất** khi hòa).

**Bất biến:** sau bước `i`, `left = Σ nums[0..i]`, `right = Σ nums[i+1..n−1]`, và `ans` là chỉ số nhỏ nhất trong `0..i` đạt hiệu nhỏ nhất.

**Có cấu trúc nào giúp bỏ bớt chỉ số không?** Hiệu trung bình **không đơn điệu** theo `i` (ví dụ 1: 3, 2, 2, 0, 1, 4) nên không thể tìm nhị phân — phải xét mọi `i`. Nhưng có thể **dừng sớm** khi gặp hiệu 0 (không thể nhỏ hơn), và đó là chỉ số nhỏ nhất đạt 0.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tiền tố đầy đủ

In [11]:
def prefix_array(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); total = pre[-1]
    best, ans = None, 0
    for i in range(n):
        a = pre[i + 1] // (i + 1)
        k = n - i - 1
        b = (total - pre[i + 1]) // k if k else 0
        d = abs(a - b)
        if best is None or d < best:
            best, ans = d, i
    return ans

for nums, exp in examples:
    got = prefix_array(nums)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp cách 2:** O(n) thời gian, O(n) bộ nhớ.

### 7.2 Cách 3 — Một lượt, hai biến tổng (cách doocs)

In [12]:
def running_sums(nums):
    n = len(nums); left, right = 0, sum(nums)
    best, ans = float("inf"), 0
    for i, x in enumerate(nums):
        left += x; right -= x
        a = left // (i + 1)
        b = right // (n - i - 1) if i < n - 1 else 0
        d = abs(a - b)
        if d < best:
            best, ans = d, i
    return ans

for nums, exp in examples:
    got = running_sums(nums)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp cách 3:** O(n) thời gian, O(1) bộ nhớ phụ.

### 7.3 Cách 4 — Một lượt + dừng sớm khi hiệu bằng 0

In [13]:
def running_early(nums):
    n = len(nums); left, right = 0, sum(nums)
    best, ans = float("inf"), 0
    for i, x in enumerate(nums):
        left += x; right -= x
        d = abs(left // (i + 1) - (right // (n - i - 1) if i < n - 1 else 0))
        if d < best:
            best, ans = d, i
            if d == 0:
                break                       # không thể nhỏ hơn 0; chỉ số này là nhỏ nhất đạt 0
    return ans

for nums, exp in examples:
    print(running_early(nums), "OK" if running_early(nums) == exp else "SAI")

3 OK
0 OK


### 7.4 Cách 5 — Biểu thức "Pythonic" với `accumulate` + `min(key=…)`

In [14]:
def pythonic(nums):
    n = len(nums); total = sum(nums)
    pre = list(accumulate(nums))
    def diff(i):
        k = n - i - 1
        return abs(pre[i] // (i + 1) - ((total - pre[i]) // k if k else 0))
    return min(range(n), key=diff)

for nums, exp in examples:
    print(pythonic(nums), "OK" if pythonic(nums) == exp else "SAI")

3 OK
0 OK


`min(range(n), key=diff)` trả chỉ số **đầu tiên** đạt min — đúng quy tắc hòa.

## 8. TRACE từng bước

Trace cách 3 trên `nums = [4, 2, 0, 6, 1, 1]` (tự dựng, đối chiếu bằng brute force).

In [15]:
def trace(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = float("inf"), 0
    print(f"total = {right}")
    for i, x in enumerate(nums):
        left += x; right -= x
        a = left // (i + 1); k = n - i - 1; b = right // k if k else 0
        d = abs(a - b); upd = d < best
        if upd: best, ans = d, i
        print(f"i={i} x={x}: left={left:>2}//{i+1}={a}, right={right:>2}//{k}={b} -> |{a}-{b}|={d}"
              f"{'  <- cập nhật' if upd else ''}  (best={best}, ans={ans})")
    return ans

print("Kết quả:", trace([4, 2, 0, 6, 1, 1]), "| brute:", brute([4, 2, 0, 6, 1, 1]))

total = 14
i=0 x=4: left= 4//1=4, right=10//5=2 -> |4-2|=2  <- cập nhật  (best=2, ans=0)
i=1 x=2: left= 6//2=3, right= 8//4=2 -> |3-2|=1  <- cập nhật  (best=1, ans=1)
i=2 x=0: left= 6//3=2, right= 8//3=2 -> |2-2|=0  <- cập nhật  (best=0, ans=2)
i=3 x=6: left=12//4=3, right= 2//2=1 -> |3-1|=2  (best=0, ans=2)
i=4 x=1: left=13//5=2, right= 1//1=1 -> |2-1|=1  (best=0, ans=2)
i=5 x=1: left=14//6=2, right= 0//0=0 -> |2-0|=2  (best=0, ans=2)
Kết quả: 2 | brute: 2


Trace ví dụ 1 để thấy dòng "cập nhật" chỉ xảy ra khi hiệu nhỏ hơn **hẳn**:

In [16]:
trace([2, 5, 3, 9, 5, 3])

total = 27
i=0 x=2: left= 2//1=2, right=25//5=5 -> |2-5|=3  <- cập nhật  (best=3, ans=0)
i=1 x=5: left= 7//2=3, right=20//4=5 -> |3-5|=2  <- cập nhật  (best=2, ans=1)
i=2 x=3: left=10//3=3, right=17//3=5 -> |3-5|=2  (best=2, ans=1)
i=3 x=9: left=19//4=4, right= 8//2=4 -> |4-4|=0  <- cập nhật  (best=0, ans=3)
i=4 x=5: left=24//5=4, right= 3//1=3 -> |4-3|=1  (best=0, ans=3)
i=5 x=3: left=27//6=4, right= 0//0=0 -> |4-0|=4  (best=0, ans=3)


3

## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def minimumAverageDifference(self, nums: List[int]) -> int:
        n = len(nums)
        left, right = 0, sum(nums)
        best, ans = float("inf"), 0
        for i, x in enumerate(nums):
            left += x                         # tổng i+1 phần tử đầu
            right -= x                        # tổng n-i-1 phần tử cuối
            a = left // (i + 1)
            b = right // (n - i - 1) if i < n - 1 else 0   # trung bình 0 phần tử = 0
            d = abs(a - b)
            if d < best:                      # '<' để giữ chỉ số nhỏ nhất khi hòa
                best, ans = d, i
        return ans

In [18]:
sol = Solution()
for nums, exp in examples:
    got = sol.minimumAverageDifference(nums)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

3 | kỳ vọng 3 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

**Tính đúng của từng giá trị.** Bất biến: ngay sau dòng `right −= x` ở vòng `i`, `left = Σ_{t ≤ i} nums[t]` và `right = Σ_{t > i} nums[t]`. Đúng lúc đầu (`left = 0`, `right = total` ứng với "trước chỉ số 0"), và mỗi vòng chuyển đúng `nums[i]` từ phải sang trái. Khi đó `a`, `b` đúng là hai trung bình làm tròn xuống theo định nghĩa (với quy ước `b = 0` khi bên phải rỗng) ⇒ `d` là hiệu trung bình của `i`.

**Tính đúng của `ans`.** Duyệt `i` tăng dần và chỉ cập nhật khi `d < best` (nhỏ hơn **ngặt**) ⇒ `ans` là chỉ số **đầu tiên** đạt giá trị nhỏ nhất. ∎

**Tại sao không bỏ sót chỉ số nào?** Vòng lặp duyệt đủ `n` chỉ số `0..n−1`; không có giả thiết đơn điệu nào được dùng (mục 4.1 cho thấy giả thiết đó sai).

**Không tràn số / không mất chính xác:** tổng tới `10⁵ · 10⁵ = 10¹⁰` — Python số nguyên tùy ý; ta không dùng `float` cho trung bình nên không có sai số làm tròn.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | O(n²) | O(n) (lát cắt) |
| 2. Mảng tiền tố | O(n) | O(n) |
| 3. Hai biến tổng | O(n) | O(1) |
| 4. + dừng sớm | O(n) worst, có thể sớm hơn | O(1) |
| 5. accumulate + min | O(n) | O(n) |

In [19]:
rng = random.Random(2256)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
for name, f in [("prefix_array", prefix_array), ("running_sums", running_sums), ("running_early", running_early),
                ("pythonic", pythonic), ("Solution", Solution().minimumAverageDifference)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<14} -> {r}  {dt*1000:.0f} ms")

prefix_array   -> 1027  19 ms
running_sums   -> 1027  16 ms
running_early  -> 1027  1 ms
pythonic       -> 1027  18 ms
Solution       -> 1027  16 ms


Dừng sớm (cách 4) có ích khi hiệu 0 xuất hiện sớm — ví dụ mảng có nhiều phần tử bằng nhau ở đầu. Đo trên hai loại dữ liệu:

In [20]:
early_hit = [5] * 10**5                              # hiệu 0 ngay chỉ số 0
no_hit = [rng.randint(0, 10**5) for _ in range(10**5)]
for name, data in [("hiệu 0 ở đầu", early_hit), ("ngẫu nhiên", no_hit)]:
    t = time.perf_counter(); running_sums(data); t1 = time.perf_counter() - t
    t = time.perf_counter(); running_early(data); t2 = time.perf_counter() - t
    print(f"{name:<14}: không dừng sớm {t1*1000:5.1f} ms | dừng sớm {t2*1000:5.1f} ms")

hiệu 0 ở đầu  : không dừng sớm  13.9 ms | dừng sớm   0.4 ms


ngẫu nhiên    : không dừng sớm  18.4 ms | dừng sớm  17.0 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {"brute": brute, "prefix_array": prefix_array, "running_sums": running_sums,
       "running_early": running_early, "pythonic": pythonic, "Solution": Solution().minimumAverageDifference}
edge = examples + [
    ([5], 0),
    ([0, 0, 0, 0], 0),               # mọi hiệu bằng 0 -> chỉ số nhỏ nhất
    ([1, 1], 0),                     # i=0: |1 - 1| = 0
    ([4, 2, 0, 6, 1, 1], None),      # so với brute
    ([10**5] * 5, 0),
    ([0, 10**5], 1),                 # i=0: |0-100000|; i=1: |50000-0|
]
for nums, exp in edge:
    if exp is None: exp = brute(nums)
    for name, f in ALL.items():
        a = list(nums)
        assert f(a) == exp, (name, nums, f(list(nums)), exp)
        assert a == nums
print("Edge case OK:", [ALL["Solution"](list(x)) for x, _ in edge])

Edge case OK: [3, 0, 0, 0, 0, 2, 0, 1]


Ghi chú các case tự dựng: `[1, 1]` — `i = 0` cho `|1 − 1| = 0` nên đáp án là 0 (dễ nhầm thành 1 nếu nghĩ "phải chia đôi giữa"); `[0, 100000]` — `i = 0` cho `|0 − 100000| = 100000`, `i = 1` cho `|50000 − 0| = 50000` ⇒ đáp án 1. Case `[4, 2, 0, 6, 1, 1]` lấy kỳ vọng từ brute force (kết quả 2, xem trace mục 8).

### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [22]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 30); hi = rng.choice([0, 1, 3, 10, 10**5])
    nums = [rng.randint(0, hi) for _ in range(n)]
    ref = brute(nums)
    for name, f in ALL.items():
        assert f(list(nums)) == ref, (name, nums)
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Kiểm tra "chỉ số nhỏ nhất khi hòa" bằng mảng toàn bằng nhau

In [23]:
for v in [0, 1, 7, 10**5]:
    for n in [1, 2, 5, 50]:
        nums = [v] * n
        # mọi i < n-1 có hiệu 0; i = n-1 có hiệu v -> đáp án 0
        assert Solution().minimumAverageDifference(nums) == 0
print("hòa: luôn trả chỉ số nhỏ nhất")

hòa: luôn trả chỉ số nhỏ nhất


### 12.4 Cực trị theo ràng buộc

- `n = 10⁵`, mọi phần tử `10⁵` ⇒ tổng `10¹⁰` (vượt `int32`); mọi hiệu bằng 0 trừ chỉ số cuối ⇒ đáp án 0.
- Mảng tăng dần `0, 1, …, n−1`: trung bình trái `≈ i/2`, phải `≈ (n + i)/2` ⇒ hiệu `≈ n/2` gần như không đổi; trung bình phải rỗng ở cuối ⇒ chỉ số cuối có hiệu `≈ (n−1)/2`. So các cách để chắc chắn.

In [24]:
big_same = [10**5] * 10**5
inc = list(range(10**5))
for name, f in ALL.items():
    if name == "brute": continue
    assert f(big_same) == 0
assert len({f(inc) for name, f in ALL.items() if name != "brute"}) == 1
print("cực trị OK; mảng tăng dần ->", Solution().minimumAverageDifference(inc), "| tổng lớn nhất:", sum(big_same))

cực trị OK; mảng tăng dần -> 99999 | tổng lớn nhất: 10000000000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia cho 0 ở chỉ số cuối

In [25]:
def wrong_div_zero(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = float("inf"), 0
    try:
        for i, x in enumerate(nums):
            left += x; right -= x
            d = abs(left // (i + 1) - right // (n - i - 1))        # SAI khi i = n-1
            if d < best: best, ans = d, i
        return ans
    except ZeroDivisionError as e:
        return f"ZeroDivisionError: {e}"

print("bản sai :", wrong_div_zero([2, 5, 3, 9, 5, 3]), "| bản đúng:", Solution().minimumAverageDifference([2, 5, 3, 9, 5, 3]))

bản sai : ZeroDivisionError: integer division or modulo by zero | bản đúng: 3


**Sửa:** trung bình của 0 phần tử là 0 ⇒ `b = right // k if k else 0`.

### Lỗi 2 — Dùng `<=` khi cập nhật (trả chỉ số **lớn nhất** khi hòa)

In [26]:
def wrong_le(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = float("inf"), 0
    for i, x in enumerate(nums):
        left += x; right -= x
        d = abs(left // (i + 1) - (right // (n - i - 1) if i < n - 1 else 0))
        if d <= best: best, ans = d, i                     # SAI
    return ans

print("bản sai :", wrong_le([0, 0, 0, 0]), "| bản đúng:", Solution().minimumAverageDifference([0, 0, 0, 0]))

bản sai : 3 | bản đúng: 0


**Sửa:** `<` (ngặt).

### Lỗi 3 — Lấy hiệu của trung bình **thực** rồi mới làm tròn

In [27]:
def wrong_round_after(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = float("inf"), 0
    for i, x in enumerate(nums):
        left += x; right -= x
        a = Fraction(left, i + 1); b = Fraction(right, n - i - 1) if i < n - 1 else Fraction(0)
        d = int(abs(a - b))                                  # SAI: làm tròn sau khi trừ
        if d < best: best, ans = d, i
    return ans

nums = [0, 1, 1]
print("bản sai :", wrong_round_after(nums), "| bản đúng:", Solution().minimumAverageDifference(nums))
for i in range(3):
    L, R = sum(nums[:i + 1]), sum(nums[i + 1:]); k = 3 - i - 1
    print(f"  i={i}: làm tròn trước |{L}//{i+1} - {R}//{k if k else '-'}| = {avg_diff(nums, i)};"
          f" làm tròn sau = {int(abs(Fraction(L, i + 1) - (Fraction(R, k) if k else 0)))}")

bản sai : 1 | bản đúng: 2
  i=0: làm tròn trước |0//1 - 2//2| = 1; làm tròn sau = 1
  i=1: làm tròn trước |1//2 - 1//1| = 1; làm tròn sau = 0
  i=2: làm tròn trước |2//3 - 0//-| = 0; làm tròn sau = 0


**Vì sao sai:** đề làm tròn **từng** trung bình trước, rồi mới lấy hiệu. `⌊a⌋ − ⌊b⌋` có thể khác `⌊a − b⌋` (lệch tối đa 1). Ở `[0, 1, 1]`, chỉ số 1 có trung bình thực `1/2` và `1`: làm tròn trước cho `|0 − 1| = 1`, làm tròn sau cho `⌊1/2⌋ = 0`. Bản sai thấy hiệu 0 ngay ở chỉ số 1 và chọn nó; theo đúng định nghĩa thì hiệu 0 đầu tiên xuất hiện ở chỉ số 2 (`|0 − 0| = 0`, bên phải rỗng). **Sửa:** `left // (i+1)` và `right // k` rồi mới trừ.

### Lỗi 4 — Nhầm số phần tử bên trái là `i` (thay vì `i + 1`)

In [28]:
def wrong_count(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); total = pre[-1]; best, ans = float("inf"), 0
    for i in range(n):
        a = pre[i] // i if i else 0                          # SAI: dùng i phần tử đầu
        k = n - i
        b = (total - pre[i]) // k if k else 0
        d = abs(a - b)
        if d < best: best, ans = d, i
    return ans

print("bản sai :", wrong_count([2, 5, 3, 9, 5, 3]), "| bản đúng:", Solution().minimumAverageDifference([2, 5, 3, 9, 5, 3]))

bản sai : 4 | bản đúng: 3


**Sửa:** bên trái **gồm cả** `nums[i]` ⇒ `i + 1` phần tử, bên phải `n − i − 1` phần tử.

### Lỗi 5 — Dùng trung bình `float` và `round`

In [29]:
def wrong_float_round(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = float("inf"), 0
    for i, x in enumerate(nums):
        left += x; right -= x
        a = round(left / (i + 1)); b = round(right / (n - i - 1)) if i < n - 1 else 0   # SAI: round != làm tròn xuống
        d = abs(a - b)
        if d < best: best, ans = d, i
    return ans

nums = [2, 5, 3, 9, 5, 3]
print("bản sai :", wrong_float_round(nums), "| bản đúng:", Solution().minimumAverageDifference(nums))

bản sai : 1 | bản đúng: 3


**Vì sao sai:** `round(3.5) = 4` (làm tròn gần nhất, kiểu "banker's rounding"), không phải 3. **Sửa:** `//`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia mảng tại mọi vị trí, cần tổng/trung bình hai phía" ⇒ **tổng tiền tố** hoặc **hai biến chạy** `left`, `right`.
- Đọc kỹ **số phần tử mỗi phía** (có gồm phần tử `i` không?) và **phía rỗng** (quy ước 0).
- "Hòa lấy chỉ số nhỏ nhất" ⇒ cập nhật bằng `<`, hoặc `min(range(n), key=…)`.
- Làm tròn: `//` (xuống), không `round`, không `int(float)` khi có số âm.
- Bài cùng họ: 724 Find Pivot Index, 1991 Find the Middle Index in Array, 2270 Number of Ways to Split Array, 2574 Left and Right Sum Differences.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trung bình **thực** (không làm tròn), so sánh chính xác bằng `Fraction`

In [30]:
def min_exact_avg_diff(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = None, 0
    for i, x in enumerate(nums):
        left += x; right -= x
        a = Fraction(left, i + 1); b = Fraction(right, n - i - 1) if i < n - 1 else Fraction(0)
        d = abs(a - b)
        if best is None or d < best: best, ans = d, i
    return ans, best

print(min_exact_avg_diff([2, 5, 3, 9, 5, 3]), min_exact_avg_diff([1, 2, 1, 2, 0]))

(3, Fraction(3, 4)) (0, Fraction(1, 4))


`Fraction` cho so sánh chính xác — tránh float coi hai số gần nhau là bằng nhau. Có thể so sánh không cần `Fraction` bằng nhân chéo: `|L·k − R·(i+1)| / ((i+1)·k)`.

### 15.2 Biến thể B — Trả về **mọi** chỉ số đạt min

In [31]:
def all_min_indices(nums):
    n = len(nums); left, right = 0, sum(nums); best = None; out = []
    for i, x in enumerate(nums):
        left += x; right -= x
        d = abs(left // (i + 1) - (right // (n - i - 1) if i < n - 1 else 0))
        if best is None or d < best: best, out = d, [i]
        elif d == best: out.append(i)
    return out

print(all_min_indices([2, 5, 3, 9, 5, 3]), all_min_indices([0, 0, 0, 0]), all_min_indices([3, 1, 2, 2]))

[3] [0, 1, 2, 3] [1, 2]


### 15.3 Biến thể C — Bên trái là `i` phần tử đầu (không gồm `nums[i]`), bên phải gồm `nums[i]`

Chỉ đổi thời điểm cập nhật `left`: tính trước rồi mới cộng. Quy ước trung bình rỗng = 0 giờ áp dụng ở **đầu** mảng (`i = 0`).

In [32]:
def variant_shift(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = float("inf"), 0
    for i, x in enumerate(nums):
        a = left // i if i else 0; b = right // (n - i)
        d = abs(a - b)
        if d < best: best, ans = d, i
        left += x; right -= x
    return ans

def variant_shift_brute(nums):
    n = len(nums)
    vals = [abs((sum(nums[:i]) // i if i else 0) - sum(nums[i:]) // (n - i)) for i in range(n)]
    return vals.index(min(vals))

rng = random.Random(153)
for _ in range(500):
    nums = [rng.randint(0, 20) for _ in range(rng.randint(1, 15))]
    assert variant_shift(nums) == variant_shift_brute(nums)
print(variant_shift([2, 5, 3, 9, 5, 3]), "- khớp brute force")

4 - khớp brute force


### 15.4 Biến thể D — Chia thành **ba** phần `[0..i]`, `[i+1..j]`, `[j+1..n−1]`, tối thiểu hóa (max TB − min TB)

Với `n` nhỏ dùng tiền tố + duyệt cặp `(i, j)` O(n²).

In [33]:
def three_parts(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); best = None
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            avgs = [pre[i + 1] // (i + 1), (pre[j + 1] - pre[i + 1]) // (j - i), (pre[n] - pre[j + 1]) // (n - j - 1)]
            v = max(avgs) - min(avgs)
            if best is None or v < best[0]: best = (v, i, j)
    return best

print(three_parts([2, 5, 3, 9, 5, 3]))

(2, 1, 2)


### 15.5 Biến thể E — Mảng có **số âm**: "trung bình làm tròn về 0" khác `//`

Nếu đề cho phép `nums[i] < 0` và định nghĩa "chia nguyên" kiểu C/Java (làm tròn **về 0**), thì `//` của Python (làm tròn **xuống**) cho kết quả khác với số âm. Phải dùng `int(a / b)` (cẩn thận float với số lớn) hoặc tự viết hàm chia về 0 bằng số nguyên:

In [34]:
def div_toward_zero(a, b):
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

def min_avg_diff_c_style(nums):
    n = len(nums); left, right = 0, sum(nums); best, ans = None, 0
    for i, x in enumerate(nums):
        left += x; right -= x
        a = div_toward_zero(left, i + 1); b = div_toward_zero(right, n - i - 1) if i < n - 1 else 0
        d = abs(a - b)
        if best is None or d < best: best, ans = d, i
    return ans

for a, b in [(-7, 2), (7, -2), (-6, 3), (5, 2)]:
    print(f"{a}/{b}: // = {a // b}, về 0 = {div_toward_zero(a, b)}")
nums = [-3, 4, -1]
print("chỉ số tốt nhất kiểu Python //:", Solution().minimumAverageDifference(nums), "| kiểu C (về 0):", min_avg_diff_c_style(nums))

-7/2: // = -4, về 0 = -3
7/-2: // = -4, về 0 = -3
-6/3: // = -2, về 0 = -2
5/2: // = 2, về 0 = 2
chỉ số tốt nhất kiểu Python //: 2 | kiểu C (về 0): 2


Bài học: khi chuyển lời giải giữa ngôn ngữ, **phép chia nguyên với số âm** là chỗ hay sai nhất. Ở bài gốc mọi số không âm nên hai kiểu trùng nhau.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | cộng lại hai lát cắt | O(n²) | O(n) | kiểm chứng |
| 2. Mảng tiền tố | `pre[i+1]`, `total − pre[i+1]` | O(n) | O(n) | nhiều truy vấn |
| 3. Hai biến chạy | `left += x`, `right −= x` | O(n) | O(1) | **mặc định** |
| 4. + dừng sớm | dừng khi hiệu 0 | O(n) | O(1) | tối ưu nhỏ |
| 5. accumulate + min | key function | O(n) | O(n) | code ngắn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumAverageDifference([1, 2, 3, 4, 5])` và `minimumAverageDifference([7, 0])`?

**Bài 2 (tính tay):** Lập bảng hiệu trung bình cho `[4, 2, 0, 6, 1, 1]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về **mảng hiệu trung bình** của mọi chỉ số (O(n)).

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` cài cách 2 (mảng tiền tố) không nhìn code.

**Bài 5 (mini challenge):** Có `q` truy vấn `(l, r)`: xét mảng con `nums[l..r]` như một mảng độc lập, trả về chỉ số (tương đối trong mảng con, bắt đầu 0) có hiệu trung bình nhỏ nhất. Viết `bt5_todo(nums, queries)` dùng một mảng tiền tố chung, O(độ dài đoạn) mỗi truy vấn.

In [35]:
def bt3_todo(nums):
    # TODO: mảng hiệu trung bình
    pass

def bt4_todo(nums):
    # TODO: mảng tiền tố
    pass

def bt5_todo(nums, queries):
    # TODO: truy vấn trên mảng con
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: total = 15. Mức 2: hiệu: `|1 − 3|=2, |1 − 4|=3, |2 − 4|=2, |2 − 5|=3, |3 − 0|=3`. Mức 3: `0`; mảng hai: `i=0: |7 − 0| = 7`, `i=1: |3 − 0| = 3` ⇒ `1`.

**Bài 2.** Mức 1: total = 14. Mức 2: `i=0: |4 − 2| = 2`, `i=1: |3 − 2| = 1`. Mức 3: `i=2: |2 − 2| = 0`, rồi `|3 − 1| = 2`, `|2 − 1| = 1`, `|2 − 0| = 2` ⇒ đáp án 2.

**Bài 3.** Mức 1: như cách 3 nhưng `append(d)`. Mức 2: nhớ `k = 0`. Mức 3: trả về list độ dài `n`.

**Bài 4.** Mức 1: `pre = accumulate(nums, initial=0)`. Mức 2: trái `pre[i+1]`, phải `total − pre[i+1]`. Mức 3: cập nhật bằng `<`.

**Bài 5.** Mức 1: trong đoạn `[l, r]`, độ dài `m = r − l + 1`. Mức 2: tổng trái của chỉ số tương đối `t` = `pre[l + t + 1] − pre[l]`. Mức 3: tổng phải = `pre[r + 1] − pre[l + t + 1]`, đếm phải `m − t − 1`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `0`, `1`. **Bài 2:** hiệu `[2, 1, 0, 2, 1, 2]` ⇒ đáp án `2`.

In [36]:
assert Solution().minimumAverageDifference([1, 2, 3, 4, 5]) == 0
assert Solution().minimumAverageDifference([7, 0]) == 1

def bt3(nums):
    n = len(nums); left, right = 0, sum(nums); out = []
    for i, x in enumerate(nums):
        left += x; right -= x
        out.append(abs(left // (i + 1) - (right // (n - i - 1) if i < n - 1 else 0)))
    return out

def bt4(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); tot = pre[-1]; best, ans = None, 0
    for i in range(n):
        k = n - i - 1
        d = abs(pre[i + 1] // (i + 1) - ((tot - pre[i + 1]) // k if k else 0))
        if best is None or d < best: best, ans = d, i
    return ans

def bt5(nums, queries):
    pre = list(accumulate(nums, initial=0)); out = []
    for l, r in queries:
        m = r - l + 1; best, ans = None, 0
        for t in range(m):
            L = pre[l + t + 1] - pre[l]; R = pre[r + 1] - pre[l + t + 1]; k = m - t - 1
            d = abs(L // (t + 1) - (R // k if k else 0))
            if best is None or d < best: best, ans = d, t
        out.append(ans)
    return out

assert bt3([4, 2, 0, 6, 1, 1]) == [2, 1, 0, 2, 1, 2]
rng = random.Random(1956)
for _ in range(1500):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 25))]
    assert bt3(nums) == [avg_diff(nums, i) for i in range(len(nums))]
    assert bt4(nums) == brute(nums)
    qs = []
    for _ in range(3):
        l = rng.randrange(len(nums)); r = rng.randrange(l, len(nums)); qs.append((l, r))
    assert bt5(nums, qs) == [brute(nums[l:r + 1]) for l, r in qs]
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
left, right = 0, sum(nums); best, ans = inf, 0
for i, x in enumerate(nums):
    left += x; right -= x
    a = left // (i + 1)
    b = right // (n - i - 1) if i < n - 1 else 0
    if abs(a - b) < best:
        best, ans = abs(a - b), i
return ans
```

**Checklist:**
- [ ] Trái gồm `i + 1` phần tử (có `nums[i]`), phải `n − i − 1`.
- [ ] Phía rỗng ⇒ trung bình 0 (không chia cho 0).
- [ ] Làm tròn **xuống** từng trung bình **trước** khi trừ (`//`).
- [ ] Cập nhật bằng `<` để lấy chỉ số nhỏ nhất.
- [ ] Đã thử: `n = 1`, toàn 0, hai phần tử.